# Introduction to Jupyter Notebook

For this hands-on lab, we use [JupyterLab](https://jupyterlab.readthedocs.io/en/stable/) to manage our environment.  The [JupyterLab Interface](https://jupyterlab.readthedocs.io/en/stable/user/interface.html) is a dashboard that provides access to interactive iPython notebooks, as well as the folder structure of our environment and a terminal window into the Ubuntu operating system. The first view you'll see includes a **menu bar** at the top, a **file browser** in the **left sidebar**, and a **main work area** that is initially open to the launcher page or the overview notebook (launcher page shown here). 

<img src="images/jl_launcher.png">

## JupyterLab File Browser
The file browser can be navigated just like any other file explorer. A double click on any of the directory items will navigate into the directory.  A double click on any file item will open a new tab (generally editable) with the file content.

## Main Work Area Layout
The main work area includes tabbed views of open files that can be closed, moved, and edited as needed. 

The notebooks, including this one, consist of a series of content and code **cells**.  To execute code in a code cell, press `Shift+Enter` or the "Run" button in the menu bar above, while a cell is highlighted. Sometimes, a content cell will get switched to editing mode. Pressing `Shift+Enter` will switch it back to a readable form.

Try executing the simple print statement in the cell below.

In [ ]:
# Highlight this cell and click [Shift+Enter] to execute
print('This is just a simple print statement')

## Jupyter Terminal
An Ubuntu terminal can be opened from the launch page.  To open a new launch page, click the small "+" sign in the upper left menu of the JupyterLab file browser.  Watch this brief clip below to see how to set up the terminal in a split screen (FYI, this video has no sound).  Try it yourself and leave the terminal available for use in the project. To copy/paste into a JupyterLab Linux terminal, copy a line with CTRL+C (or CMD+C in Mac), and then RIGHT-CLICK to paste onto a terminal command line.

<img src="images/getting_started_jupyter.png" alt="getting started" style="width:65%;">


# Download the Notebooks

For your convenience, we have collected the juypter notebooks, scripts, and configurations into a zip file.  Download the file by right clicking on `nim_workshop.zip` .
<img src="images/download_zip.png" alt="getting started" style="width:35%;">



# API KEY CHECK

![build.nvidia.com](images/build_nvidia.png)

In this notebook, we will make sure our environment is setup correctly with API keys.  This will make the lab go smoother and ensure we can access the models we need.



# NGC Authentication and API Keys

There are two ways to access models on build.nvidia.com 
1.  Get a key from build.nvidia.com 
2.  Setup an API key from NGC

For this lab we will be using a pre-authorized key from NGC.  The following is some supplementary information on how to get an API key from NGC.

**Supplementary Information**

In the [NGC Home Screen](https://ngc.nvidia.com/), use the top-right dropdown to selct the org/team with access to the NIM containers.

You can navigate to your required team by clicking your account button on the top right corner of the website. Then click on the first option in the drop down list (which shows the name of your current team), and navigate to the team.

An NGC API key is required to access NGC resources and a key can be generated here: https://org.ngc.nvidia.com/setup/personal-keys.

When creating an NGC API key, ensure that “NGC Catalog” and "Private Registry" is selected from the “Services Included” dropdown. More Services can be included if this key is to be reused for other purposes.

![image](images/ngc_key.png)

This API key will need to be passed to `docker run` in the next section as the `NGC_API_KEY` environment variable to download the appropriate models and resources when starting the NIM. So please remember to store this key somewhere safe, to reuse for later.

In [ ]:
# Stop all prior running containers
!scripts/stop_containers.sh

### Import Environment

In [ ]:
# Setup Environment
import os
from dotenv import load_dotenv
# Load the .env file
load_dotenv('./docker/.env', override=True)

print(f"NGC_API_KEY for lab use only     : {os.getenv('NGC_API_KEY')}")
print(f"NVIDIA_API_KEY for lab use only  : {os.getenv('NVIDIA_API_KEY')}")
print(f"Current working dir              : { os.getcwd() }")
print(f"PROJECT_DIR for lab              : {os.getenv('PROJECT_DIR')}")
print(f"LOCAL_NIM_CACHE for lab          : {os.getenv('LOCAL_NIM_CACHE')}")
print(f"locally hosted model for lab     : {os.getenv('NIM_SERVED_MODEL_NAME')}")


Your output should look similar like this (keys might be different but should start with nvapi)

```code
NGC_API_KEY for lab use only     : nvapi-xxxxxxxxxxxx
NVIDIA_API_KEY for lab use only  : nvapi-xxxxxxxxxxxx
Current working dir              : /home/nvidia/nim-workshop
PROJECT_DIR for lab              : /home/nvidia/nim-workshop
LOCAL_NIM_CACHE for lab          : /home/nvidia/nim-workshop/.cache
locally hosted model for lab     : mistralai/mistral-7b-instruct-v0.3
```

In [ ]:
# Quick API Test -- just run and make sure you get a response, will explain in the subsequent notebook 
from openai import OpenAI

client = OpenAI(
  base_url = "https://integrate.api.nvidia.com/v1",
  api_key  = os.getenv('NVIDIA_API_KEY')
)

completion = client.chat.completions.create(
  model="nvdev/mistralai/mistral-7b-instruct-v0.3",
  messages=[{"role":"user","content":"this is a quick test. tell me something random, but keep it short."}],
  temperature=0.2,
  top_p=0.7,
  max_tokens=20,
  stream=True
)

for chunk in completion:
  if chunk.choices[0].delta.content is not None:
    print(chunk.choices[0].delta.content, end="")

Verify that the notebook finished without error.  If error, will see a long error trace.

---

# Supplementary Information 
* Use the following information to setup a local environment.  No need to run anything here.

### Install the NGC CLI if needed

  - **Note:** you may need to install [NGC CLI](https://org.ngc.nvidia.com/setup/installers/cli) in order to pull the LoRA weights from NGC to run the above command and export as a local variale to use it. 
  - You will also need to make sure that the NGC CLI is locally in the same parent folder for sit to run. Next you will need to log into the NGC CLI. Then export PATH if there are still issues,
  - If this is not possible with your permissions, please feel free to skip this step.

 How to setup ngc cli --> https://org.ngc.nvidia.com/setup/installers/cli
 1. get version that matches your system
 2. Make the NGC CLI binary executable and add your current directory to path:
```bash
    !chmod u+x ngc-cli/ngc
    !echo "export PATH=\"\$PATH:$(pwd)/ngc-cli\"" >> ~/.bash_profile && source ~/.bash_profile
```

- Download fine tuned math model<br>
**Note: this is already be in your Launchpad, so we will move it to our loras directory**

Sample ngc download
```bash
  !ngc registry model download-version --dest loras "nim/meta/llama3-8b-instruct-lora:nemo-math-v1"
```

-----

### Install and login to Nvidia Docker CLI


https://docs.nvidia.com/datacenter/cloud-native/container-toolkit/latest/install-guide.html

Configure the production repository:
```bash
curl -fsSL https://nvidia.github.io/libnvidia-container/gpgkey | sudo gpg --dearmor -o /usr/share/keyrings/nvidia-container-toolkit-keyring.gpg \
  && curl -s -L https://nvidia.github.io/libnvidia-container/stable/deb/nvidia-container-toolkit.list | \
    sed 's#deb https://#deb [signed-by=/usr/share/keyrings/nvidia-container-toolkit-keyring.gpg] https://#g' | \
    sudo tee /etc/apt/sources.list.d/nvidia-container-toolkit.list

```bash
sudo apt-get update
sudo apt-get install -y nvidia-container-toolkit


Prerequisites:
-   You installed a supported container engine (Docker, Containerd, CRI-O, Podman).

-   You installed the NVIDIA Container Toolkit.

Configuring Docker:
Configure the container runtime by using the nvidia-ctk command:
```bash
sudo nvidia-ctk runtime configure --runtime=docker
```
The nvidia-ctk command modifies the `/etc/docker/daemon.json` file on the host. The file is updated so that Docker can use the NVIDIA Container Runtime.

Restart the Docker daemon:
```bash
sudo systemctl restart docker
```
Run a sample CUDA container:
```bash
sudo docker run --rm --runtime=nvidia --gpus all ubuntu nvidia-smi
```


If you run `docker login nvcr.io` (as detailed in NVIDIA docs)
you will be prompted. Then you can use `$oauthtoken` as the username and your[ NGC API Key](https://org.ngc.nvidia.com/setup/personal-keys) as the password.  You will only need to do this once and then it will be added to your cache.

For a single command, you can use the following:
```bash 
bash -c 'echo "$NVIDIA_API_KEY" | docker login -u $oauthtoken --password-stdin nvcr.io'
```

Using the interactive login method:
```bash
$ docker login nvcr.io

Username: $oauthtoken
Password: $NGC_API_KEY
```

---